# Class 05 · Part 2: Evaluating a model

In part 1, `class05_get_healpix_data.ipynb`, you saved the monthly 2 m temperature of 2021–2023 from the
ICON simulation `ngc4008` and from ERA5, both on the HEALPix grid at zoom 6. Now you **evaluate** the model:

1. **maps** of both,
2. the **bias**: model minus ERA5, cell by cell,
3. the **global mean**, as a time series, and
4. the time series at **one place**, Hamburg, together with the station of class 02.

## 1. Packages and data

The saved files open in a second. We convert from K to °C.

In [ ]:
import cartopy.crs as ccrs
import easygems.healpix as egh
import healpix as hp
import matplotlib.pyplot as plt
import xarray as xr

icon = xr.open_dataarray("data/icon_ngc4008_tas_monthly_2021-2023_zoom6.nc") - 273.15
era5 = xr.open_dataarray("data/era5_2t_monthly_2021-2023_zoom6.nc") - 273.15

print(icon.sizes, era5.sizes)

Same number of months, same cells: both are on the same grid, so you can subtract them, as in class 04.

## 2. Maps

The mean of the 36 months, side by side, with the same colours:

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(16, 4), subplot_kw={"projection": ccrs.Robinson()}
)
for ax, data, name in [(axes[0], icon, "ICON ngc4008"), (axes[1], era5, "ERA5")]:
    ax.set_global()
    ax.coastlines(linewidth=0.5)
    image = egh.healpix_show(
        data.mean("time"), ax=ax, cmap="RdYlBu_r", vmin=-30, vmax=30
    )
    ax.set_title(f"{name}, 2 m temperature, 2021–2023")
fig.colorbar(image, ax=axes, shrink=0.7, label="temperature / °C")

fig.savefig("/app/pages/05_healpix/maps.png", dpi=100, bbox_inches="tight")
plt.show()

They look almost the same: the big pattern, warm tropics and cold poles, is right in the model. The
interesting part is the **difference**.

## 3. The bias

The **bias** is model minus observation: positive where the model is too warm, negative where it is too cold.

In [ ]:
bias = icon.mean("time") - era5.mean("time")

fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.Robinson())
ax.set_global()
ax.coastlines(linewidth=0.5)
image = egh.healpix_show(bias, ax=ax, cmap="RdBu_r", vmin=-5, vmax=5)
fig.colorbar(image, ax=ax, shrink=0.7, label="ICON − ERA5 / K")
ax.set_title("Bias of ICON ngc4008, 2 m temperature, 2021–2023")

fig.savefig("/app/pages/05_healpix/bias.png", dpi=100, bbox_inches="tight")
plt.show()

Where is the model too warm, where too cold? Look at the continents of the north, the tropics, and Antarctica.

## 4. The global mean

All HEALPix cells have the **same area**, so the global mean is a plain `.mean("cell")`: no weights, unlike
the lon-lat grid of class 03.

In [ ]:
icon_global = icon.mean("cell")
era5_global = era5.mean("cell")

print(f"global mean bias: {float((icon_global - era5_global).mean()):+.2f} K")

fig, ax = plt.subplots(figsize=(10, 4))
icon_global.plot(ax=ax, marker="o", label="ICON ngc4008")
era5_global.plot(ax=ax, marker="o", label="ERA5")
ax.set_ylabel("temperature / °C")
ax.set_xlabel("month")
ax.set_title("Global mean 2 m temperature")
ax.legend()

fig.savefig("/app/pages/05_healpix/timeseries_global.png", dpi=100, bbox_inches="tight")
plt.show()

## 5. One place: Hamburg

A HEALPix cell has a number, not a latitude and longitude. `hp.ang2pix` gives the number of the cell that
contains a position (extra 01 uses it too). `nside` is $2^6 = 64$ at zoom 6, and `nest=True` is the order of
the cells in these datasets.

In [ ]:
nside = hp.npix2nside(icon.sizes["cell"])  # 64 at zoom 6
hamburg = hp.ang2pix(nside, 10.0, 53.55, nest=True, lonlat=True)  # longitude, latitude
print("nside", nside, "| cell of Hamburg:", hamburg)

And the station of class 02, Hamburg-Fuhlsbüttel: its monthly means of the same months. A station is a
point, a cell of zoom 6 is about 100 km wide, so do not expect them to be equal.

In [ ]:
import zipfile

import pandas as pd

from utils import download_dwd_daily

zip_file = zipfile.ZipFile(download_dwd_daily(1975))
data_name = [
    name for name in zip_file.namelist() if name.startswith("produkt_klima_tag")
][0]
station = pd.read_csv(
    zip_file.open(data_name), sep=";", skipinitialspace=True, na_values=[-999]
)
station["date"] = pd.to_datetime(station["MESS_DATUM"].astype(str), format="%Y%m%d")
station_monthly = (
    station.set_index("date")["TMK"].resample("MS").mean().loc["2021":"2023"]
)

fig, ax = plt.subplots(figsize=(10, 4))
icon.isel(cell=hamburg).plot(ax=ax, marker="o", label="ICON ngc4008")
era5.isel(cell=hamburg).plot(ax=ax, marker="o", label="ERA5")
station_monthly.plot(ax=ax, marker="o", color="black", label="DWD station")
ax.set_ylabel("temperature / °C")
ax.set_xlabel("month")
ax.set_title("Hamburg, monthly mean 2 m temperature")
ax.legend(loc="upper left", ncol=3)

fig.savefig(
    "/app/pages/05_healpix/timeseries_hamburg.png", dpi=100, bbox_inches="tight"
)
plt.show()

ERA5 and the station follow each other month by month: ERA5 is made with the observations of the real
weather (class 07). The model does not, and is not expected to: it was started in 2020 and then makes its
**own weather**. So in a single month it can be several degrees off, warmer or colder. What tells you about
the model is the mean over many months, as in the bias map: the **climate**, not the weather.

**Try it:** how large is the bias in Hamburg, month by month? `(icon - era5).isel(cell=hamburg)`. Is it as
regular as the global one? And its mean over the 36 months?

In [ ]:
(icon - era5).isel(cell=hamburg).round(1).to_series()

## 6. Exercise: a second simulation

Evaluate **`ngc3028`**, saved in the exercise of part 1:

1. Its bias map, saved as `/app/pages/05_healpix/bias_exercise.png`.
2. Its global mean together with ngc4008 and ERA5 (three curves), saved as
   `/app/pages/05_healpix/timeseries_exercise.png`.
3. Which simulation is closer to ERA5, globally and in the north (e.g. Siberia, Canada)?

In [ ]:
# 1. the bias of ngc3028

In [ ]:
# 2. three curves of the global mean

## 7. Your page

Open `pages/05_healpix/my_page.rst` in JupyterLab, write your name, and describe the biases you found.
Look at it on http://localhost:8000 → *Classes → 05*.